# Testing 
* **The Four Questions:** The specification explicitly lists four question types for the BraTS 2021 task:
1. Location             2. Size                3. Sub region presence             4. Multifocality
  
### **Cell 1: Initialization & Loading the VLM**

Run this cell to load your frozen base models and inject your trained adapter weights.

In [ ]:
import os
import torch
import glob
import tarfile
import shutil
import numpy as np
import nibabel as nib
import scipy.ndimage as ndimage
from nltk.translate.bleu_score import sentence_bleu

# 1. Load the Model & Your Saved Adapter
print("🔄 Initializing Mixture-of-Encoders Medical VLM...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model (This automatically loads BrainIAC and LLaMA as frozen layers)
model = BrainTumorVLM()

# Load ONLY the trainable FFN adapter weights you just trained
checkpoint_path = "/kaggle/input/notebooks/aliqaiser1123/training-vlm/brain_tumor_adapter.pt"
if os.path.exists(checkpoint_path):
    model.adapter.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("✅ Successfully loaded trained adapter weights!")
else:
    print("⚠️ Warning: adapter.pt not found. Using untrained adapter for testing.")

model.eval()

# 2. Preprocessing Function
def preprocess_flair_scan(flair_path, target_shape=(96, 96, 96)):
    """Preprocesses a .nii.gz file into the 96^3 tensor expected by BrainIAC."""
    flair_data = nib.load(flair_path).get_fdata()
    factors = [t / s for t, s in zip(target_shape, flair_data.shape)]
    resized = ndimage.zoom(flair_data, factors, order=1)
    mask = resized > 0
    if np.any(mask):
        mean = np.mean(resized[mask])
        std = np.std(resized[mask]) + 1e-8
        resized[mask] = (resized[mask] - mean) / std
    return torch.tensor(resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 3. Inference Function
def ask_vlm(model, mri_tensor, question):
    """Passes the scan and question through the VLM data flow."""
    target_device = next(model.llm.parameters()).device
    target_dtype = torch.bfloat16

    mri_tensor = mri_tensor.to(device=target_device, dtype=torch.float32)

    with torch.no_grad():
        # f_vision(x) -> Z
        vit_out = model.vision_encoder(mri_tensor)
        if isinstance(vit_out, tuple): vit_out = vit_out[0]

        # g_i(Z) -> H_img
        h_img = model.adapter(vit_out).to(dtype=target_dtype)

        # LLM_embed(question) -> E_q
        prompt = f"Question: {question}\nAnswer:"
        prompt_ids = model.tokenizer(prompt, return_tensors="pt").input_ids.to(target_device)
        e_prompt = model.llm.get_input_embeddings()(prompt_ids).to(dtype=target_dtype)

        # Concat: H = [ H_img ; E_q ]
        inputs_embeds = torch.cat([h_img, e_prompt], dim=1)

        # LLM decoder generates the answer
        generated_ids = model.llm.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=50,
            pad_token_id=model.tokenizer.eos_token_id,
            do_sample=False # Greedy decoding for factual medical answers
        )

        return model.tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()

print("✅ Setup complete. Ready for evaluation.")

---

### **Cell 2: Testing the 4 Target Questions & Evaluation Logic**

This cell extracts a test scan, asks the 4 specific questions defined in your architecture, and demonstrates how to evaluate the responses against a ground truth.

In [ ]:
# 1. Extract a single patient scan for testing
tar_path = glob.glob("/kaggle/input/**/brats2021_processed.tar", recursive=True)[0]
temp_dir = "/tmp/eval_patient"
os.makedirs(temp_dir, exist_ok=True)

print("📦 Extracting test patient data...")
with tarfile.open(tar_path, "r") as tf:
    for member in tf.getmembers():
        if "BraTS2021_00005" in member.name: # Use a specific test patient
            tf.extract(member, path=temp_dir)

flair_file = glob.glob(f"{temp_dir}/**/*_flair.nii.gz", recursive=True)[0]
mri_tensor = preprocess_flair_scan(flair_file)

# 2. Define the 4 Question Types per your architecture spec
# In a real evaluation loop, 'ground_truth' comes from your dataset engine.
qa_pairs = [
    {
        "type": "Location",
        "question": "In which brain hemisphere and lobe is the primary tumor mass located?",
        "ground_truth": "The tumor is primarily located in the right frontal lobe."
    },
    {
        "type": "Size",
        "question": "What is the approximate solid tumor volume in cubic millimeters?",
        "ground_truth": "The approximate solid tumor volume is 24500 mm3."
    },
    {
        "type": "Sub region presence",
        "question": "Is peritumoral edema present in this scan?",
        "ground_truth": "Yes, peritumoral edema is present."
    },
    {
        "type": "Multifocality",
        "question": "Does the scan show evidence of multifocal tumor growth?",
        "ground_truth": "No, there is a single focal mass."
    }
]

print("\n🏥 --- VLM INFERENCE & EVALUATION --- 🏥\n")

total_bleu = 0

for item in qa_pairs:
    print(f"📌 Question Type: {item['type']}")
    print(f"❓ Question: {item['question']}")

    # Generate Answer
    prediction = ask_vlm(model, mri_tensor, item['question'])

    print(f"🎯 Ground Truth: {item['ground_truth']}")
    print(f"🤖 VLM Prediction: {prediction}")

    # Simple NLP Evaluation (BLEU Score)
    reference = [item['ground_truth'].lower().split()]
    candidate = prediction.lower().split()

    # Calculate BLEU-1 (word overlap) for basic metric demonstration
    score = sentence_bleu(reference, candidate, weights=(1.0, 0, 0, 0))
    total_bleu += score

    print(f"📊 BLEU-1 Score: {score:.4f}\n")
    print("-" * 50 + "\n")

average_bleu = total_bleu / len(qa_pairs)
print(f"🏆 Average BLEU-1 Score for this scan: {average_bleu:.4f}")

# Cleanup
shutil.rmtree(temp_dir, ignore_errors=True)